In [3]:
# install dependencies
# !pip install tensorflow==2.3.0
# !pip install keras==2.4.3
# !pip install gym==0.17.3
# !pip install keras-rl2==1.0.4


In [4]:
from typing import Optional
import numpy as np
import gymnasium as gym


class GridWorldEnv(gym.Env):

    def __init__(self, size: int = 5):
        # The size of the square grid (5x5 by default)
        self.size = size

        # Initialize positions - will be set randomly in reset()
        # Using -1,-1 as "uninitialized" state
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # Define what the agent can observe
        # Dict space gives us structured, human-readable observations
        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),   # [x, y] coordinates
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),  # [x, y] coordinates
            }
        )

        # Define what actions are available (4 directions)
        self.action_space = gym.spaces.Discrete(4)

        # Map action numbers to actual movements on the grid
        # This makes the code more readable than using raw numbers
        self._action_to_direction = {
            0: np.array([1, 0]),   # Move right (positive x)
            1: np.array([0, 1]),   # Move up (positive y)
            2: np.array([-1, 0]),  # Move left (negative x)
            3: np.array([0, -1]),  # Move down (negative y)
        }
    
    def _get_obs(self):
        """Convert internal state to observation format.
    
        Returns:
            dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location, "target": self._target_location}
    
    def _get_info(self):
        """Compute auxiliary information for debugging.

        Returns:
            dict: Info with distance between agent and target
        """
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location, ord=1
            )
        }

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info

    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-3 for directions)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        # Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

        # Update agent position, ensuring it stays within grid bounds
        # np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
        truncated = False

        # Simple reward structure: +1 for reaching target, 0 otherwise
        # Alternative: could give small negative rewards for each step to encourage efficiency
        reward = 1 if terminated else 0

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info



In [7]:
import gym



import random


In [4]:
from gym import Env
from gym.spaces import Discrete, Box
import numpy as np
import matplotlib.pyplot as plt

class PulsrEnv(Env):
    def __init__(self):
        # Actions: 0=N, 1=E, 2=S, 3=W
        self.action_space = Discrete(4)
        
        # State: [x, y, score]
        self.observation_space = Box(
            low=np.array([-100.0, -100.0, 0.0], dtype=np.float32),
            high=np.array([100.0, 100.0, np.inf], dtype=np.float32),
            dtype=np.float32
        )
        
        # Circle radius for the ball trajectory
        self.radius = 100
        self.max_steps = 239
        
        # Visualization setup
        self.fig, self.ax = None, None
        self.agent_plot, self.ball_plot, self.traj_plot = None, None, None
        self.trajectory = []
        
        self.reset()
    
    def step(self, action):
        old_x, old_y, _ = self.state
        
        # --- Apply action ---
        if action == 0:   # North
            self.state[1] += 1
        elif action == 1: # East
            self.state[0] += 1
        elif action == 2: # South
            self.state[1] -= 1
        elif action == 3: # West
            self.state[0] -= 1
        
        # --- Keep agent inside workspace ---
        if not (-100 <= self.state[0] <= 100 and -100 <= self.state[1] <= 100):
            self.state[0], self.state[1] = old_x, old_y
        
        # --- Move ball along circular trajectory ---
        angle = (2 * np.pi * self.episode_length) / self.max_steps
        self.ball_pos = np.array([
            self.radius * np.sin(angle),   # x
            self.radius * np.cos(angle)    # y (starts at top)
        ])
        
        # --- Distance & scoring ---
        d = np.linalg.norm(self.state[:2] - self.ball_pos)
        if d <= 70:
            self.state[2] += 1   # increase score
        
        # --- Use score as reward ---
        reward = self.state[2]
        
        # --- Step update ---
        self.episode_length += 1
        done = self.episode_length >= self.max_steps
        
        # --- Final reward mapping ---
        info = {}
        if done:
            score = self.state[2]
            if score > 150:
                info["final_reward"] = 3
            elif 100 <= score <= 150:
                info["final_reward"] = 2
            elif 90 <= score < 100:
                info["final_reward"] = 1
            elif score == 90:
                info["final_reward"] = 0
            elif 70 <= score < 90:
                info["final_reward"] = -1
            elif 50 <= score < 70:
                info["final_reward"] = -2
            else:
                info["final_reward"] = -3
        
        return self.state.copy(), reward, done, info
        
    def reset(self):
        # Reset agent and trajectory
        self.state = np.array([0, 0, 0], dtype=np.float32)
        self.episode_length = 0
        self.trajectory = []
        
        # Start ball at angle=0
        self.ball_pos = np.array([self.radius, 0])
        
        return self.state.copy()
    
    def render(self, mode="human"):
        if self.fig is None:
            plt.ion()
            self.fig, self.ax = plt.subplots(figsize=(6, 6));
            # plt.close(self.fig)   # suppress <Figure ...> in notebook output
            
            self.ax.set_xlim(-120, 120)
            self.ax.set_ylim(-120, 120)
            self.ax.set_aspect("equal")
            self.ax.set_title("PULSR Environment")
            
            circle = plt.Circle((0, 0), self.radius, color="gray", fill=False, linestyle="--")
            self.ax.add_patch(circle)
            
            self.agent_plot, = self.ax.plot([], [], "bo", label="Agent")
            self.ball_plot, = self.ax.plot([], [], "ro", label="Ball")
            self.traj_plot, = self.ax.plot([], [], "b--", linewidth=0.8, alpha=0.6, label="Trajectory")
            
            self.ax.legend()

        # Update agent and ball positions
        self.agent_plot.set_data([self.state[0]], [self.state[1]])
        self.ball_plot.set_data([self.ball_pos[0]], [self.ball_pos[1]])
        
        if len(self.trajectory) > 1:
            traj = np.array(self.trajectory)
            self.traj_plot.set_data(traj[:, 0], traj[:, 1])
        
        plt.draw()
        plt.pause(0.01)


In [5]:
env = PulsrEnv()

In [6]:
env.observation_space.sample()

array([  4.1280737, -29.678541 ,   0.5121868], dtype=float32)

In [7]:
episodes = 10
for episode in range(1, episodes+1):
    state = env.reset()
    done = False
    score = 0 
    
    while not done:
        # env.render()
        action = env.action_space.sample()
        n_state, reward, done, info = env.step(action)
        # print(n_state)
        # score = reward
    print('Episode:{} Score:{} info:{}'.format(episode, reward, info))

Episode:1 Score:0.0 info:{'final_reward': -3}
Episode:2 Score:0.0 info:{'final_reward': -3}
Episode:3 Score:0.0 info:{'final_reward': -3}
Episode:4 Score:0.0 info:{'final_reward': -3}
Episode:5 Score:0.0 info:{'final_reward': -3}
Episode:6 Score:0.0 info:{'final_reward': -3}
Episode:7 Score:0.0 info:{'final_reward': -3}
Episode:8 Score:0.0 info:{'final_reward': -3}
Episode:9 Score:0.0 info:{'final_reward': -3}
Episode:10 Score:0.0 info:{'final_reward': -3}


### Create A Deep Learning Model with Keras

In [59]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam

In [60]:
states = env.observation_space.shape
actions = env.action_space.n

In [61]:
actions

4

In [62]:
def build_model(states, actions):
    model = Sequential()    
    model.add(Dense(24, activation='relu', input_shape=states))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(actions, activation='linear'))
    return model

In [63]:
model = build_model(states, actions)

C:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [64]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 796 (3.11 KB)

 Trainable params: 796 (3.11 KB)

 Non-trainable params: 0 (0.00 B)

### Build Agent with Keras-RL

In [67]:
# from rl.agents import DQNAgent
# from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [68]:
!pip install stable-baselines3


In [70]:
from stable_baselines3 import DQN
from stable_baselines3.dqn import MlpPolicy



In [73]:
# Create the DQN agent
model = DQN(
    "MlpPolicy",       # replaces the Keras model + policy
    env,               # your PulsrEnv
    learning_rate=1e-3,
    buffer_size=50000, # replaces SequentialMemory
    learning_starts=10, # replaces nb_steps_warmup
    target_update_interval=100, # similar to target_model_update
    verbose=1
)

# Train the agent
model.learn(total_timesteps=50000)

# Save model
# model.save("dqn_pulsr")

# Reload model if needed
# model = DQN.load("dqn_pulsr", env=env)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


C:\Users\user\anaconda3\Lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 239      |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.818    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 669      |
|    time_elapsed     | 1        |
|    total_timesteps  | 956      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.00758  |
|    n_updates        | 236      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 239      |
|    ep_rew_mean      | 350      |
|    exploration_rate | 0.637    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 564      |
|    time_elapsed     | 3        |
|    total_timesteps  | 1912     |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 5.95     |
|    n_updates      

In [72]:
!pip install shimmy

NameError: name 'dqn' is not defined